In [0]:
print("Versión de Spark:", spark.version)

Versión de Spark: 4.2.0


In [0]:
%fs ls /Volumes/workspace/ef_precios_medicamentos/datos/sinteticos_1m.csv

path,name,size,modificationTime
dbfs:/Volumes/workspace/ef_precios_medicamentos/datos/sinteticos_1m.csv,sinteticos_1m.csv,240356190,1795310969000


In [0]:
from pyspark.sql import functions as F
import time

ruta = "/Volumes/workspace/ef_precios_medicamentos/datos/sinteticos_1m.csv"
df = spark.read.csv(ruta, header=True, inferSchema=True)
print("Filas:", df.count())
df.printSchema()

Filas: 1000000
root
 |-- codigo_producto: double (nullable = true)
 |-- nombre_comercial: string (nullable = true)
 |-- concentracion: string (nullable = true)
 |-- forma_farmaceutica: string (nullable = true)
 |-- presentacion: string (nullable = true)
 |-- registro_sanitario: string (nullable = true)
 |-- ifa: string (nullable = true)
 |-- titular: string (nullable = true)
 |-- fabricante: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- clasificacion: string (nullable = true)
 |-- n_establecimientos: double (nullable = true)
 |-- precio_media: double (nullable = true)
 |-- precio_mediana: double (nullable = true)
 |-- precio_moda: double (nullable = true)
 |-- periodo: date (nullable = true)
 |-- mes: integer (nullable = true)
 |-- nombre_mes: string (nullable = true)
 |-- trimestre: integer (nullable = true)
 |-- anio: integer (nullable = true)
 |-- variacion_mensual_pct: double (nullable = true)



In [0]:
#dato extra del informe
originales = df.withColumn("p", F.spark_partition_id()).select("p").distinct().count()
print("Particiones al leer el CSV:", originales)

Particiones al leer el CSV: 8


In [0]:
def medir(nombre, accion, repeticiones=3):
    # El primer comando (calentamiento) NO se registra
    accion()
    tiempos = []
    for i in range(repeticiones):
        t0 = time.perf_counter()
        accion()
        tiempos.append(time.perf_counter() - t0)
    tiempos_ord = sorted(tiempos)
    print(nombre, "│ tiempos (s):", [round(t, 2) for t in tiempos],
          "│ mediana (s):", round(tiempos_ord[len(tiempos_ord) // 2], 2))
    return tiempos

resultados = {}

def experimento(n):
    dfn = df.repartition(n)
    reales = dfn.withColumn("p", F.spark_partition_id()).select("p").distinct().count()
    print("repartition", n, "-> particiones reales:", reales)
    tiempos = medir(f"repartition_{n}",
                    lambda: dfn.groupBy("clasificacion", "pais").agg(F.avg("precio_media"), F.count("*")).collect())
    resultados[n] = (reales, *[round(t, 2) for t in tiempos], round(sorted(tiempos)[1], 2))

In [0]:
experimento(1)

repartition 1 -> particiones reales: 1
repartition_1 │ tiempos (s): [1.09, 1.0, 1.0] │ mediana (s): 1.0


In [0]:
experimento(2)

repartition 2 -> particiones reales: 2
repartition_2 │ tiempos (s): [1.42, 1.17, 1.01] │ mediana (s): 1.17


In [0]:
experimento(4)

repartition 4 -> particiones reales: 4
repartition_4 │ tiempos (s): [1.06, 0.91, 0.94] │ mediana (s): 0.94


In [0]:
experimento(8) 

repartition 8 -> particiones reales: 8
repartition_8 │ tiempos (s): [0.94, 0.89, 0.89] │ mediana (s): 0.89


In [0]:
experimento(16)

repartition 16 -> particiones reales: 16
repartition_16 │ tiempos (s): [0.93, 0.92, 0.91] │ mediana (s): 0.92


In [0]:
experimento(32)

repartition 32 -> particiones reales: 32
repartition_32 │ tiempos (s): [0.95, 0.95, 0.94] │ mediana (s): 0.95


In [0]:
filas = [(n, *v) for n, v in sorted(resultados.items())]
display(spark.createDataFrame(filas, ["particiones_pedidas", "particiones_reales", "rep1_s", "rep2_s", "rep3_s", "mediana_s"]))

particiones_pedidas,particiones_reales,rep1_s,rep2_s,rep3_s,mediana_s
1,1,1.09,1.0,1.0,1.0
2,2,1.42,1.17,1.01,1.17
4,4,1.06,0.91,0.94,0.94
8,8,0.94,0.89,0.89,0.89
16,16,0.93,0.92,0.91,0.92
32,32,0.95,0.95,0.94,0.95


Databricks visualization. Run in Databricks to view.